# 1. Inistialisation

In [21]:
from pyspark.sql import functions as F
from pyspark.ml.feature import (
    VectorAssembler,
    StringIndexer,
    OneHotEncoder
)
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

print("🚀 Démarrage Notebook 7 — ML Modeling")


🚀 Démarrage Notebook 7 — ML Modeling


# Probleme N°1: Prédire la durée d’un trajet taxi

#### ➡️ Régression
#### ➡️ KPI business : temps de trajet, ETA, pricing dynamique

## 2. Chargement du Feature Store

In [16]:
features = spark.table("iceberg.gold.ml_trip_features")

print(f"📊 Nombre de lignes: {features.count():,}")
features.printSchema()


📊 Nombre de lignes: 67,510,043
root
 |-- trip_duration_minutes: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- is_morning: boolean (nullable = true)
 |-- is_evening: boolean (nullable = true)
 |-- is_night: boolean (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- is_same_borough: boolean (nullable = true)
 |-- is_manhattan_trip: boolean (nullable = true)
 |-- is_airport_trip: boolean (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- real_distance_miles: double (nullable = true)
 |-- temp: double (nullab

## 3. Sélection des features ML

### target

In [17]:
target = "trip_duration_minutes"


### Features numériques

In [18]:
numeric_features = [
    "trip_distance",
    "real_distance_miles",
    "hour_of_day",
    "day_of_week",
    "week_of_year",
    "passenger_count",
    "fare_per_mile",
    "fare_per_minute",
    "temp",
    "rhum",
    "prcp",
    "wspd",
    "recent_trips_in_zone",
    "avg_recent_duration",
    "avg_recent_fare"
]


### Features catégorielles

In [19]:
categorical_features = [
    "pickup_borough",
    "dropoff_borough"
]


In [21]:
from pyspark.sql.functions import when, col, isnan

def sanitize_numeric(df, cols):
    for c in cols:
        df = df.withColumn(
            c,
            when(isnan(col(c)) | col(c).isNull() | (col(c) == float("inf")) | (col(c) == float("-inf")), 0.0)
            .otherwise(col(c))
        )
    return df
features = sanitize_numeric(features, numeric_features)

## 4. Préparation des features (Pipeline)

In [22]:
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_features
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe"
    )
    for c in categorical_features
]

assembler = VectorAssembler(
    inputCols=numeric_features + [f"{c}_ohe" for c in categorical_features],
    outputCol="features"
)


# 5. Train / Test split

In [23]:
train_df = features.filter(F.col("year") < 2025)
test_df  = features.filter(F.col("year") >= 2025)

print(f"📊 Train: {train_df.count():,}")
print(f"📊 Test : {test_df.count():,}")


📊 Train: 35,491,352
📊 Test : 32,018,691


## Modèle 1 — Baseline Linear Regression

In [48]:

assembler = VectorAssembler(
    inputCols=numeric_features + [f"{c}_ohe" for c in categorical_features],
    outputCol="features",
    handleInvalid="keep"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol=target,
    regParam=0.01
)

pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, lr])

model_lr = pipeline_lr.fit(train_df)
pred_lr = model_lr.transform(test_df)



26/01/14 12:51:28 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
                                                                                

In [49]:
pred_lr.select(
    target,
    "prediction"
).show(10)


+---------------------+------------------+
|trip_duration_minutes|        prediction|
+---------------------+------------------+
|   44.766666666666666| 55.52870251011493|
|   43.916666666666664| 41.27525466530639|
|   18.666666666666668|32.091475358372065|
|                29.85| 33.00883130466745|
|    85.31666666666666|  69.3751910686319|
|                 42.4| 60.76456507279394|
|   142.73333333333332| 71.80240111178692|
|   20.133333333333333| 33.84702131843674|
|   39.766666666666666|50.086637141384585|
|    8.983333333333333|28.872793363581664|
+---------------------+------------------+
only showing top 10 rows



In [50]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(pred_lr)
print("RMSE:", rmse)


[Stage 200:===================================================>   (14 + 1) / 15]

RMSE: 6.692523265538405


## Modèle 2 — RANDOM FOREST


In [54]:
rf = RandomForestRegressor(
    labelCol=target,
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    seed=42
)


In [55]:
pipeline_rf = Pipeline(
    stages=indexers + encoders + [assembler, rf]
)


In [56]:
model_rf = pipeline_rf.fit(train_df)

26/01/14 13:17:54 WARN MemoryStore: Not enough space to cache rdd_516_1 in memory! (computed 770.8 MiB so far)
26/01/14 13:17:54 WARN BlockManager: Persisting block rdd_516_1 to disk instead.
26/01/14 13:18:15 WARN MemoryStore: Not enough space to cache rdd_516_1 in memory! (computed 770.8 MiB so far)
26/01/14 13:18:24 WARN MemoryStore: Not enough space to cache rdd_516_2 in memory! (computed 228.4 MiB so far)
26/01/14 13:18:24 WARN BlockManager: Persisting block rdd_516_2 to disk instead.
26/01/14 13:18:47 WARN MemoryStore: Not enough space to cache rdd_516_2 in memory! (computed 228.4 MiB so far)
26/01/14 13:18:50 WARN MemoryStore: Not enough space to cache rdd_516_3 in memory! (computed 770.8 MiB so far)
26/01/14 13:18:50 WARN BlockManager: Persisting block rdd_516_3 to disk instead.
26/01/14 13:19:04 WARN MemoryStore: Not enough space to cache rdd_516_3 in memory! (computed 770.8 MiB so far)
26/01/14 13:19:11 WARN MemoryStore: Not enough space to cache rdd_516_4 in memory! (compute

In [57]:
pred_rf = model_rf.transform(test_df)

In [58]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="rmse"
)

rmse_rf = evaluator.evaluate(pred_rf)
print("🌲 RMSE Random Forest:", rmse_rf)


[Stage 235:===================================================>   (14 + 1) / 15]

🌲 RMSE Random Forest: 3.4134986277553683


In [61]:
print("🌲 RMSE Random Forest:", rmse_rf)
print("🌲 RMSE LR:", rmse)

🌲 RMSE Random Forest: 3.4134986277553683
🌲 RMSE LR: 6.692523265538405


In [64]:
rf_model = model_rf.stages[-1]

importances = rf_model.featureImportances.toArray()

feature_names = numeric_features.copy()
for c in categorical_features:
    feature_names.append(c)

importance_df = spark.createDataFrame(
    zip(feature_names, importances.tolist()),
    ["feature", "importance"]
).orderBy(F.desc("importance"))

importance_df.show(20, truncate=False)


[Stage 237:>                                                        (0 + 2) / 2]

+--------------------+---------------------+
|feature             |importance           |
+--------------------+---------------------+
|trip_distance       |0.34003839796747687  |
|fare_per_minute     |0.22669082055106482  |
|real_distance_miles |0.16681985235696645  |
|fare_per_mile       |0.12319462569903875  |
|avg_recent_duration |0.0721444651517126   |
|dropoff_borough     |0.01858897186541356  |
|avg_recent_fare     |0.014810377821901595 |
|pickup_borough      |0.00798596167335716  |
|hour_of_day         |0.006201298259895426 |
|day_of_week         |4.132503568625978E-4 |
|week_of_year        |2.8817767388066907E-4|
|rhum                |7.00388781495635E-5  |
|passenger_count     |3.5534339114951E-5   |
|temp                |3.017471447935548E-5 |
|wspd                |2.2483227611639505E-5|
|prcp                |9.797202291130718E-6 |
|recent_trips_in_zone|1.8082253323595226E-6|
+--------------------+---------------------+



In [67]:
from pyspark.ml import PipelineModel
import shutil
import os

model_path = "/models/rf_trip_duration"

# Nettoyage si déjà existant (important en dev)
if os.path.exists(model_path):
    shutil.rmtree(model_path)

# Sauvegarde
model_rf.write().overwrite().save(model_path)

print(f"✅ Modèle Random Forest sauvegardé dans {model_path}")


26/01/14 16:49:49 WARN TaskSetManager: Stage 285 contains a task of very large size (4128 KiB). The maximum recommended task size is 1000 KiB.


✅ Modèle Random Forest sauvegardé dans /models/rf_trip_duration


# Probleme N°2: Prédiction de la DEMANDE taxi par zone & par heure

## 🎯 Objectif Business

👉 Prédire le nombre de courses à venirpar zone × heure pour :

- 🚕 repositionner les chauffeurs

- 📈 anticiper les pics de demande

- 💰 activer le surge pricing

- 🧠 aide à la décision opérationnelle

In [86]:
from pyspark.sql import functions as F

trips = spark.table("iceberg.silver.trips_complete")

hourly_demand = (
    trips
    .withColumn("hour_ts", F.date_trunc("hour", "tpep_pickup_datetime"))
    .groupBy("pickup_zone", "hour_ts")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("trip_duration_minutes").alias("avg_duration"),
        F.avg("trip_distance").alias("avg_distance")
    )
)


In [87]:
from pyspark.sql.window import Window

w = Window.partitionBy("pickup_zone").orderBy("hour_ts")

features = (
    hourly_demand
    .withColumn("trip_count_next_hour", F.lead("trip_count").over(w))
    .filter("trip_count_next_hour IS NOT NULL")
)


In [88]:
features = (
    features
    .withColumn("hour", F.hour("hour_ts"))
    .withColumn("day_of_week", F.dayofweek("hour_ts"))
    .withColumn("is_weekend", F.col("day_of_week").isin(1,7))
)


In [90]:
weather = spark.table("iceberg.gold.weather_impact")

In [91]:
weather.limit(5).toPandas()

,pickup_date,is_rainy,is_cold,total_trips,avg_distance,avg_duration,avg_fare,avg_speed,avg_tip_pct,avg_temp,avg_rhum,avg_prcp,avg_wspd,avg_pres,weather_condition,year,month
0,2024-04-03,True,True,51923,2.966809,15.729487,27.791724,10.204000,12.956287,4.702213,92.199334,1.625307,36.096874,1002.182501,rainy_cold,2024,4
1,2024-04-04,False,False,51308,3.142697,15.961560,28.761328,10.448796,12.744592,6.867311,66.636509,0.000000,23.145611,996.254795,dry_warm,2024,4
2,2024-04-03,True,False,34486,3.027516,19.730058,29.153267,8.288242,12.091485,6.051267,84.412805,0.906043,30.924114,1007.809053,rainy_warm,2024,4
3,2024-04-11,False,False,73790,3.162466,16.842973,28.460175,10.126024,12.630380,12.090785,92.713457,0.000000,14.562298,1014.549614,dry_warm,2024,4
4,2024-04-12,False,False,69965,3.325340,17.128071,28.927417,10.201829,12.355129,13.543357,83.615508,0.000000,27.853670,996.525334,dry_warm,2024,4


In [98]:
spark.table("iceberg.gold.weather_impact").printSchema()


root
 |-- pickup_date: date (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- is_rainy: boolean (nullable = true)
 |-- is_cold: boolean (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- avg_duration: double (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- avg_tip_pct: double (nullable = true)
 |-- avg_temp: double (nullable = true)
 |-- avg_rhum: double (nullable = true)
 |-- avg_prcp: double (nullable = true)
 |-- avg_wspd: double (nullable = true)
 |-- avg_pres: double (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [99]:
weather = spark.table("iceberg.gold.weather_impact")

features = (
    features
    .join(
        weather,
        F.date_trunc("hour", weather.tpep_pickup_datetime) == features.hour_ts,
        "left"
    )
)


In [105]:
features.limit(5).toPandas()

,pickup_zone,hour_ts,trip_count,avg_duration,avg_distance,trip_count_next_hour,hour,day_of_week,is_weekend,pickup_date,tpep_pickup_datetime,is_rainy,is_cold,total_trips,avg_distance,avg_duration,avg_fare,avg_speed,avg_tip_pct,avg_temp,avg_rhum,avg_prcp,avg_wspd,avg_pres,weather_condition,year,month
0,None,2024-01-01 02:00:00,19,13.134211,2.246842,19,2,2,False,2024-01-01,2024-01-01 02:19:43,False,True,3,1.806667,8.038889,19.116667,13.328777,16.655795,5.0,55.0,0.0,13.0,1016.0,dry_cold,2024,1
1,None,2024-01-01 02:00:00,19,13.134211,2.246842,19,2,2,False,2024-01-01,2024-01-01 02:01:59,False,True,1,2.130000,9.233333,17.100000,13.841155,0.000000,5.0,55.0,0.0,13.0,1016.0,dry_cold,2024,1
2,None,2024-01-01 02:00:00,19,13.134211,2.246842,19,2,2,False,2024-01-01,2024-01-01 02:26:11,False,True,1,0.380000,4.333333,10.800000,5.261538,0.000000,5.0,55.0,0.0,13.0,1016.0,dry_cold,2024,1
3,None,2024-01-01 02:00:00,19,13.134211,2.246842,19,2,2,False,2024-01-01,2024-01-01 02:54:26,False,True,5,3.208000,15.826667,26.292000,12.041963,12.232143,5.0,55.0,0.0,13.0,1016.0,dry_cold,2024,1
4,None,2024-01-01 02:00:00,19,13.134211,2.246842,19,2,2,False,2024-01-01,2024-01-01 02:48:15,False,True,3,3.626667,17.894444,31.266667,12.839759,19.444444,5.0,55.0,0.0,13.0,1016.0,dry_cold,2024,1


In [106]:
features.select(
    "hour_ts",
    "avg_temp",
    "avg_prcp",
    "avg_wspd"
).show(5)


[Stage 466:>                                                        (0 + 1) / 1]

+-------------------+--------+--------+--------+
|            hour_ts|avg_temp|avg_prcp|avg_wspd|
+-------------------+--------+--------+--------+
|2024-01-01 01:00:00|     5.0|     0.0|     9.0|
|2024-01-01 01:00:00|     5.0|     0.0|     9.0|
|2024-01-01 01:00:00|     5.0|     0.0|     9.0|
|2024-01-01 01:00:00|     5.0|     0.0|     9.0|
|2024-01-01 01:00:00|     5.0|     0.0|     9.0|
+-------------------+--------+--------+--------+
only showing top 5 rows



In [107]:
train = features.filter("hour_ts < '2025-01-01'")
test  = features.filter("hour_ts >= '2025-01-01'")


In [108]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

cat_cols = ["pickup_zone"]
num_cols = [
    "trip_count",
    "avg_duration",
    "avg_distance",
    "hour",
    "day_of_week",
    "temp",
    "prcp"
]

indexer = StringIndexer(
    inputCol="pickup_zone",
    outputCol="pickup_zone_idx",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=num_cols + ["pickup_zone_idx"],
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_count_next_hour",
    numTrees=50,
    maxDepth=10
)

pipeline = Pipeline(stages=[indexer, assembler, rf])
model = pipeline.fit(train)


IllegalArgumentException: temp does not exist. Available: pickup_zone, hour_ts, trip_count, avg_duration, avg_distance, trip_count_next_hour, hour, day_of_week, is_weekend, pickup_date, tpep_pickup_datetime, is_rainy, is_cold, total_trips, avg_distance, avg_duration, avg_fare, avg_speed, avg_tip_pct, avg_temp, avg_rhum, avg_prcp, avg_wspd, avg_pres, weather_condition, year, month, pickup_zone_idx